# Braking Analysis: Pressure, Release & Balance

This notebook provides deep analysis of braking technique and G utilization across corners.

## What You'll Find Here

- **Peak Brake Pressure Consistency**: Are you applying the same peak pressure each lap?
- **Entry Speed Consistency**: Speed at the braking point — are you arriving at the same speed?
- **Braking Distance Consistency**: Total stopping distance variability per corner
- **Brake Release Point Consistency**: Where you release the brake — key for trail braking
- **Brake Balance Analysis**: Front vs rear brake bias per corner (requires separate F/R channels)
- **G Utilization Analysis**: How continuously you use available grip through corner complexes
- **Summary Statistics Table**: All braking metrics in one table

## How to Interpret the Results

- **Tight box plots**: Consistent technique — you're braking the same way each lap
- **Wide box plots**: Inconsistent — opportunity for improvement
- **G utilization close to 100%**: Smooth transitions, minimal wasted grip
- **G utilization below 70%**: Significant grip holes in transitions (high priority coaching area)

## Requirements

- GPS data channels (`GPS Latitude`, `GPS Longitude`, `GPS Speed`)
- Brake pressure (`BrakePress`) and throttle (`PPS`)
- Lateral acceleration (`LateralAcc`) for G utilization analysis
- Separate front/rear brake channels for balance analysis (optional)

**Note:** This notebook works in both JupyterLite (browser) and standard JupyterLab environments.

In [ ]:
# Install required packages (needed for JupyterLite, skipped in regular JupyterLab if already installed)
%pip install -q pandas plotly libxrk libibt motorsports-data-notebook jinja2 ipywidgets

# Use the Rust parser backend for ~3x faster file loading
import os

os.environ["LIBXRK_BACKEND"] = "rust"

# Import core libraries
import numpy as np
import pandas as pd
from IPython.display import display

# Visualization libraries
import plotly.express as px
import plotly.graph_objects as go

# Import helper functions
from motorsports_data_notebook.channels import (
    get_best_lap_channels,
    get_top_laps,
)
from motorsports_data_notebook.corners import identify_corners
from motorsports_data_notebook.visualization import (
    format_lap_time,
    plot_track_segments,
    show_fig,
)
from motorsports_data_notebook.widgets import SessionPicker
from motorsports_data_notebook.zones import (
    compute_g_utilization,
    compute_segment_stats,
    create_track_segments,
    detect_zones_averaged,
)

# Session picker with channel configuration
# Upload your own .xrk/.xrz/.ibt file or use the sample data
session = SessionPicker(
    default_file="../data/CMD_Inferno 86_Fuji GP Sh_Generic testing_a_2248.xrz",
    channel_mapping={
        # GPS channels (required for corner detection)
        "gps_latitude": "GPS Latitude",
        "gps_longitude": "GPS Longitude",
        "gps_speed": "GPS Speed",  # Speed in m/s from GPS
        # Pedal inputs (required for zone detection)
        "throttle": "PPS",  # Throttle position sensor (0-100%)
        "brake": "BrakePress",  # Brake pressure (0-100%)
        # Dynamics (required for G utilization analysis)
        "lateral_g": "LateralAcc",  # Lateral acceleration in G
        "steering": "SteerAngle",  # Steering angle in degrees
    },
)
session.display()

In [ ]:
# Get the loaded session data
log = session.get_log()
laps = session.get_laps()
CHANNEL_NAMES = session.get_channel_names()

# Display lap times table
laps.style.format({"lap_time": format_lap_time})  # type: ignore[dict-item]

In [ ]:
# Extract best lap channel data and detect corners
best_lap, channels = get_best_lap_channels(
    log, laps, [CHANNEL_NAMES["gps_latitude"], CHANNEL_NAMES["gps_longitude"], "distance_m"]
)

best_lap_num = int(best_lap["num"])
gps_lat_ch = CHANNEL_NAMES["gps_latitude"]
gps_lon_ch = CHANNEL_NAMES["gps_longitude"]
aligned = (
    log.filter_by_lap(best_lap_num)
    .select_channels([gps_lat_ch, gps_lon_ch, "distance_m"])
    .resample_to_channel(gps_lat_ch)
    .channels
)

lap_channels = {
    "GPS Latitude": aligned[gps_lat_ch].column(gps_lat_ch).to_numpy(),
    "GPS Longitude": aligned[gps_lon_ch].column(gps_lon_ch).to_numpy(),
    "distance_m": aligned["distance_m"].column("distance_m").to_numpy(),
}

corners = identify_corners(
    lat=lap_channels["GPS Latitude"],
    lon=lap_channels["GPS Longitude"],
    threshold=0.003,
    min_corner_length=15,
    min_gap=80,
)
print(f"Found {len(corners)} corners")

In [ ]:
# Get top laps and detect zones
top_laps = get_top_laps(laps, threshold_pct=1.03)
braking_zones, accel_zones = detect_zones_averaged(log, top_laps, CHANNEL_NAMES)

track_length = lap_channels["distance_m"][-1]
segments = create_track_segments(corners, braking_zones, accel_zones, track_length)

# Compute per-lap segment statistics
stats_df = compute_segment_stats(log, top_laps, segments, CHANNEL_NAMES)

print(f"Analyzing {len(top_laps)} laps (within 103% of best time)")
print(f"Found {len(braking_zones)} braking zones, {len(accel_zones)} acceleration zones")
print(f"Computed {len(stats_df)} segment statistics across all laps")

In [ ]:
# Peak brake pressure consistency
# Shows whether the driver applies the same peak brake pressure each lap

braking_with_peak = stats_df[stats_df["segment_type"] == "braking"].dropna(subset=["peak_brake"])

if len(braking_with_peak) > 0:
    fig = px.box(
        braking_with_peak,
        x="segment_name",
        y="peak_brake",
        title="Peak Brake Pressure Consistency by Corner",
        labels={"peak_brake": "Peak Brake Pressure", "segment_name": "Corner"},
    )
    fig.update_layout(xaxis_tickangle=-45, width=900, height=500)
    show_fig(fig)
else:
    print("No peak brake pressure data available")

In [ ]:
# Entry speed consistency
# Speed at the braking point — inconsistent entry speed + consistent braking point = speed varies before braking zone

braking_with_entry = stats_df[stats_df["segment_type"] == "braking"].dropna(subset=["entry_speed"])

if len(braking_with_entry) > 0:
    fig = px.box(
        braking_with_entry,
        x="segment_name",
        y="entry_speed",
        title="Entry Speed Consistency by Corner",
        labels={"entry_speed": "Entry Speed (km/h)", "segment_name": "Corner"},
    )
    fig.update_layout(xaxis_tickangle=-45, width=900, height=500)
    show_fig(fig)
else:
    print("No entry speed data available")

In [ ]:
# Braking distance consistency
# Shows total stopping distance variability per corner

braking_with_dist = stats_df[stats_df["segment_type"] == "braking"].dropna(
    subset=["braking_distance"]
)

if len(braking_with_dist) > 0:
    fig = px.box(
        braking_with_dist,
        x="segment_name",
        y="braking_distance",
        title="Braking Distance Consistency by Corner",
        labels={"braking_distance": "Braking Distance (m)", "segment_name": "Corner"},
    )
    fig.update_layout(xaxis_tickangle=-45, width=900, height=500)
    show_fig(fig)
else:
    print("No braking distance data available")

In [ ]:
# Brake release point consistency
# Shows variation in where the driver releases the brake

braking_with_release = stats_df[stats_df["segment_type"] == "braking"].dropna(
    subset=["brake_release_point"]
)

if len(braking_with_release) > 0:
    braking_with_release = braking_with_release.copy()
    braking_with_release["release_deviation"] = braking_with_release.groupby("segment_name")[
        "brake_release_point"
    ].transform(lambda x: x - x.mean())

    fig = px.box(
        braking_with_release,
        x="segment_name",
        y="release_deviation",
        title="Brake Release Point Consistency by Corner (Centered on Mean)",
        labels={"release_deviation": "Deviation from Mean (m)", "segment_name": "Corner"},
    )
    fig.update_layout(xaxis_tickangle=-45, width=900, height=500)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
    show_fig(fig)
else:
    print("No brake release point data available")

In [ ]:
# Brake balance analysis (only available with separate front/rear brake channels)
brake_rear_ch = CHANNEL_NAMES.get("brake_rear", "")
brake_front_ch = CHANNEL_NAMES["brake"]

if brake_rear_ch:
    balance_data = []
    for idx, lap in top_laps.iterrows():
        lap_num = int(lap["num"])
        try:
            aligned = (
                log.filter_by_lap(lap_num)
                .select_channels(["distance_m", brake_front_ch, brake_rear_ch])
                .resample_to_channel("distance_m")
                .channels
            )
        except Exception:
            continue

        dist_arr = aligned["distance_m"].column("distance_m").to_numpy()
        front_arr = aligned[brake_front_ch].column(brake_front_ch).to_numpy()
        rear_arr = aligned[brake_rear_ch].column(brake_rear_ch).to_numpy()

        for seg in segments:
            if seg.segment_type != "braking":
                continue
            mask = (dist_arr >= seg.start_dist) & (dist_arr <= seg.end_dist)
            if not mask.any():
                continue
            front_peak = float(np.max(front_arr[mask]))
            rear_peak = float(np.max(rear_arr[mask]))
            total = front_peak + rear_peak
            if total > 0:
                balance_data.append(
                    {
                        "segment_name": seg.name,
                        "corner_id": seg.corner_id,
                        "lap_num": lap_num,
                        "front_peak": front_peak,
                        "rear_peak": rear_peak,
                        "front_bias_pct": front_peak / total * 100,
                    }
                )

    if balance_data:
        balance_df = pd.DataFrame(balance_data)

        # Front vs rear peak brake pressure per corner
        balance_grouped = (
            balance_df.groupby("segment_name")[["front_peak", "rear_peak"]].mean().reset_index()
        )
        fig = go.Figure()
        fig.add_trace(
            go.Bar(name="Front", x=balance_grouped["segment_name"], y=balance_grouped["front_peak"])
        )
        fig.add_trace(
            go.Bar(name="Rear", x=balance_grouped["segment_name"], y=balance_grouped["rear_peak"])
        )
        fig.update_layout(
            barmode="group",
            title="Front vs Rear Peak Brake Pressure by Corner",
            xaxis_title="Corner",
            yaxis_title="Peak Brake Pressure",
            xaxis_tickangle=-45,
            width=900,
            height=500,
        )
        show_fig(fig)

        # Front bias % box plot
        fig = px.box(
            balance_df,
            x="segment_name",
            y="front_bias_pct",
            title="Front Brake Bias by Corner (%)",
            labels={"front_bias_pct": "Front Bias (%)", "segment_name": "Corner"},
        )
        fig.update_layout(xaxis_tickangle=-45, width=900, height=500)
        fig.add_hline(
            y=50,
            line_dash="dash",
            line_color="gray",
            opacity=0.5,
            annotation_text="50% = Equal balance",
        )
        show_fig(fig)

        # Overall summary
        print(
            f"Overall front brake bias: {balance_df['front_bias_pct'].mean():.1f}% \u00b1 {balance_df['front_bias_pct'].std():.1f}%"
        )
    else:
        print("No brake balance data computed")
else:
    print("Rear brake channel not configured \u2014 brake balance analysis skipped")

In [ ]:
# Brake balance track map (only if rear brake channel available)
if brake_rear_ch:
    try:
        aligned = (
            log.filter_by_lap(best_lap_num)
            .select_channels(["distance_m", brake_front_ch, brake_rear_ch])
            .resample_to_channel("distance_m")
            .channels
        )
        dist_arr = aligned["distance_m"].column("distance_m").to_numpy()
        front_arr = aligned[brake_front_ch].column(brake_front_ch).to_numpy()
        rear_arr = aligned[brake_rear_ch].column(brake_rear_ch).to_numpy()

        total_brake = front_arr + rear_arr
        braking_mask = total_brake > np.max(total_brake) * 0.05
        balance_pct = np.where(
            braking_mask, front_arr / np.maximum(total_brake, 1e-6) * 100, np.nan
        )

        gps_dist = lap_channels["distance_m"]
        braking_indices = np.where(braking_mask)[0]
        if len(braking_indices) > 0:
            braking_dists = dist_arr[braking_indices]
            braking_balance = balance_pct[braking_indices]
            lat_interp = np.interp(braking_dists, gps_dist, lap_channels["GPS Latitude"])
            lon_interp = np.interp(braking_dists, gps_dist, lap_channels["GPS Longitude"])

            fig = go.Figure()
            fig.add_trace(
                go.Scattermapbox(
                    lat=lap_channels["GPS Latitude"],
                    lon=lap_channels["GPS Longitude"],
                    mode="markers",
                    marker=dict(size=3, color="lightgray"),
                    name="Track",
                    showlegend=False,
                )
            )
            fig.add_trace(
                go.Scattermapbox(
                    lat=lat_interp,
                    lon=lon_interp,
                    mode="markers",
                    marker=dict(
                        size=8,
                        color=braking_balance,
                        colorscale="RdYlBu_r",
                        showscale=True,
                        colorbar=dict(title="Front Bias %"),
                        cmin=40,
                        cmax=80,
                    ),
                    name="Brake Balance",
                )
            )
            fig.update_layout(
                mapbox=dict(
                    style="open-street-map",
                    center=dict(
                        lat=np.mean(lap_channels["GPS Latitude"]),
                        lon=np.mean(lap_channels["GPS Longitude"]),
                    ),
                    zoom=14,
                ),
                title="Brake Balance Track Map (Best Lap)",
                showlegend=False,
                width=800,
                height=600,
            )
            show_fig(fig)
        else:
            print("No significant braking detected in best lap")
    except Exception as e:
        print(f"Could not compute brake balance track map: {e}")
else:
    print("Rear brake channel not configured \u2014 brake balance track map skipped")

## G Utilization Analysis

G utilization measures how continuously the driver uses available tire grip through the braking \u2192 turn-in \u2192 mid-corner \u2192 exit \u2192 acceleration sequence. Dips in total G (= \u221a(lateral_g\u00b2 + inline_g\u00b2)) indicate wasted grip \u2014 coasting between braking and turning, or hesitating between turning and accelerating.

- **High g_utilization_pct** (close to 100%): Smooth transitions, minimal grip wasted
- **Low g_utilization_pct**: \"Grip holes\" in transitions \u2014 the driver pauses between phases
- **total_g_min_phase**: Which phase the grip hole occurs in \u2014 tells you *where* the driver hesitates

Requires lateral G data. Inline (longitudinal) G is derived from speed if not available.

In [ ]:
# Compute G utilization per corner per lap
# Requires lateral G; inline G derived from speed if not available

lateral_g_ch = CHANNEL_NAMES["lateral_g"]
inline_g_ch = CHANNEL_NAMES.get("inline_g", "")
gps_speed_ch = CHANNEL_NAMES["gps_speed"]

g_util_channels = ["distance_m", gps_speed_ch, lateral_g_ch]
if inline_g_ch:
    g_util_channels.append(inline_g_ch)

distances_list: list[np.ndarray] = []
speeds_list: list[np.ndarray] = []
lateral_gs_list: list[np.ndarray] = []
inline_gs_list: list[np.ndarray] | None = [] if inline_g_ch else None
lap_nums_list: list[int] = []

for idx, lap in top_laps.iterrows():
    lap_num = int(lap["num"])
    try:
        aligned = (
            log.filter_by_lap(lap_num)
            .select_channels(g_util_channels)
            .resample_to_channel("distance_m")
            .channels
        )
    except Exception:
        continue

    dist_arr = aligned["distance_m"].column("distance_m").to_numpy()
    speed_arr = aligned[gps_speed_ch].column(gps_speed_ch).to_numpy() * 3.6  # m/s -> km/h
    lat_g_arr = aligned[lateral_g_ch].column(lateral_g_ch).to_numpy()

    distances_list.append(dist_arr)
    speeds_list.append(speed_arr)
    lateral_gs_list.append(lat_g_arr)
    lap_nums_list.append(lap_num)

    if inline_g_ch and inline_gs_list is not None:
        inline_gs_list.append(aligned[inline_g_ch].column(inline_g_ch).to_numpy())

if len(distances_list) > 0:
    g_util_df = compute_g_utilization(
        distances=distances_list,
        speeds=speeds_list,
        lateral_gs=lateral_gs_list,
        inline_gs=inline_gs_list,
        lap_nums=lap_nums_list,
        segments=segments,
        corners=corners,
    )
    print(f"Computed G utilization for {len(g_util_df)} corner/lap combinations")
else:
    g_util_df = pd.DataFrame()
    print("No lateral G data available for G utilization analysis")

In [ ]:
# G Utilization by Corner — box plot
# Shows how consistently the driver maintains grip through each corner complex

if len(g_util_df) > 0:
    corner_order = [c.name for c in corners if c.name in g_util_df["corner_name"].values]
    fig = px.box(
        g_util_df,
        x="corner_name",
        y="g_utilization_pct",
        title="G Utilization by Corner (Higher = Smoother Transitions)",
        labels={
            "g_utilization_pct": "G Utilization (%)",
            "corner_name": "Corner",
        },
        category_orders={"corner_name": corner_order},
    )
    fig.update_layout(xaxis_tickangle=-45, width=900, height=500)
    fig.add_hline(
        y=70,
        line_dash="dash",
        line_color="orange",
        opacity=0.5,
        annotation_text="70% = Significant grip holes",
    )
    show_fig(fig)
else:
    print("No G utilization data available")

In [ ]:
# Phase G Breakdown — grouped bar chart
# Shows mean total G per phase per corner, highlighting where the driver leaves grip on the table

if len(g_util_df) > 0:
    phase_cols = ["braking_g_mean", "entry_g_mean", "mid_g_mean", "exit_g_mean"]
    phase_labels = ["Braking", "Entry", "Mid-corner", "Exit"]

    # Average across laps per corner
    phase_avg = g_util_df.groupby("corner_name")[phase_cols].mean().reset_index()

    corner_order = [c.name for c in corners if c.name in phase_avg["corner_name"].values]
    phase_avg = phase_avg.set_index("corner_name").loc[corner_order].reset_index()

    fig = go.Figure()
    colors = ["#EF553B", "#FFA15A", "#00CC96", "#636EFA"]
    for col, label, color in zip(phase_cols, phase_labels, colors):
        fig.add_trace(
            go.Bar(
                name=label,
                x=phase_avg["corner_name"],
                y=phase_avg[col],
                marker_color=color,
            )
        )

    fig.update_layout(
        barmode="group",
        title="Mean Total G by Phase per Corner (Lowest Phase = Grip Hole)",
        xaxis_title="Corner",
        yaxis_title="Mean Total G",
        xaxis_tickangle=-45,
        width=900,
        height=500,
    )
    show_fig(fig)
else:
    print("No G utilization data available for phase breakdown")

In [ ]:
# Total G Trace — best lap overlay with corner shading and G hole markers
# Shows total G vs distance for the best lap with corner regions highlighted

if len(g_util_df) > 0:
    gps_speed_ch = CHANNEL_NAMES["gps_speed"]

    bl_channels = ["distance_m", gps_speed_ch, lateral_g_ch]
    if inline_g_ch:
        bl_channels.append(inline_g_ch)

    bl_aligned = (
        log.filter_by_lap(best_lap_num)
        .select_channels(bl_channels)
        .resample_to_channel("distance_m")
        .channels
    )

    bl_dist = bl_aligned["distance_m"].column("distance_m").to_numpy()
    bl_speed = bl_aligned[gps_speed_ch].column(gps_speed_ch).to_numpy() * 3.6
    bl_lat_g = bl_aligned[lateral_g_ch].column(lateral_g_ch).to_numpy()

    if inline_g_ch:
        bl_inl_g = bl_aligned[inline_g_ch].column(inline_g_ch).to_numpy()
    else:
        # Derive from speed
        spd_ms = bl_speed / 3.6
        dd = np.diff(bl_dist)
        avg_spd = (spd_ms[:-1] + spd_ms[1:]) / 2
        safe_avg = np.where(avg_spd > 0.1, avg_spd, 0.1)
        dt = dd / safe_avg
        dv = np.diff(spd_ms)
        safe_dt = np.where(dt > 1e-6, dt, 1e-6)
        accel = dv / safe_dt / 9.81
        accel = np.concatenate([[accel[0]], accel])
        kernel = np.ones(5) / 5
        bl_inl_g = np.convolve(accel, kernel, mode="same")

    bl_total_g = np.sqrt(bl_lat_g**2 + bl_inl_g**2)

    fig = go.Figure()

    # Add corner shading
    corner_colors = ["rgba(100,100,255,0.1)", "rgba(255,100,100,0.1)"]
    for i, corner in enumerate(corners):
        fig.add_vrect(
            x0=corner.start_dist,
            x1=corner.end_dist,
            fillcolor=corner_colors[i % 2],
            layer="below",
            line_width=0,
            annotation_text=corner.name,
            annotation_position="top left",
            annotation_font_size=9,
        )

    # Total G trace
    fig.add_trace(
        go.Scatter(
            x=bl_dist,
            y=bl_total_g,
            mode="lines",
            name="Total G",
            line=dict(color="#636EFA", width=1.5),
        )
    )

    # Add G hole markers from best lap g_util_df
    best_lap_g = g_util_df[g_util_df["lap_num"] == best_lap_num]
    if "total_g_min_dist" in best_lap_g.columns and len(best_lap_g) > 0:
        fig.add_trace(
            go.Scatter(
                x=best_lap_g["total_g_min_dist"],
                y=best_lap_g["total_g_min"],
                mode="markers+text",
                marker=dict(size=10, color="red", symbol="diamond"),
                text=[
                    f"{row['corner_name']}: {row['total_g_min']:.2f}G"
                    for _, row in best_lap_g.iterrows()
                ],
                textposition="top center",
                textfont=dict(size=9, color="red"),
                name="G Holes",
            )
        )

    fig.update_layout(
        title=f"Total G Trace — Best Lap (Lap {best_lap_num})",
        xaxis_title="Distance (m)",
        yaxis_title="Total G",
        width=1100,
        height=400,
        showlegend=False,
    )
    show_fig(fig)
else:
    print("No G utilization data available for total G trace")

In [ ]:
# G Hole Detection — identify corners where total G drops significantly during transitions
# A G hole indicates the driver is not smoothly overlapping braking and turning (trail braking gap)
# or turning and accelerating (exit hesitation)

if len(g_util_df) > 0 and "total_g_min_dist" in g_util_df.columns:
    # Average G hole metrics across laps per corner
    g_hole_summary = (
        g_util_df.groupby("corner_name")
        .agg(
            g_min_mean=("total_g_min", "mean"),
            g_min_std=("total_g_min", "std"),
            g_max_mean=("total_g_max", "mean"),
            g_util_mean=("g_utilization_pct", "mean"),
            g_min_dist_mean=("total_g_min_dist", "mean"),
        )
        .reset_index()
    )

    # Compute G hole depth: how far the min drops below the peak
    g_hole_summary["g_hole_depth"] = g_hole_summary["g_max_mean"] - g_hole_summary["g_min_mean"]

    # Determine most common phase for each corner
    from collections import Counter

    phase_map = {}
    for name, group in g_util_df.groupby("corner_name"):
        phases = group["total_g_min_phase"].dropna().values
        if len(phases) > 0:
            phase_map[name] = Counter(phases).most_common(1)[0][0]
    g_hole_summary["g_hole_phase"] = g_hole_summary["corner_name"].map(phase_map)

    # Sort by G hole depth (worst first)
    g_hole_summary = g_hole_summary.sort_values("g_hole_depth", ascending=False)

    # Flag significant G holes
    g_hole_summary["severity"] = g_hole_summary["g_util_mean"].apply(
        lambda x: "HIGH" if x < 30 else ("MEDIUM" if x < 50 else ("LOW" if x < 70 else "OK"))
    )

    # Phase interpretation
    phase_advice = {
        "braking": "G hole during braking — late/abrupt brake application",
        "entry": "Trail braking gap — brakes released before turn-in builds lateral G",
        "mid": "Mid-corner hesitation — driver lifts or pauses at apex",
        "exit": "Exit hesitation — gap between turning and accelerating",
    }
    g_hole_summary["interpretation"] = g_hole_summary["g_hole_phase"].map(phase_advice)

    # Display
    display_cols = [
        "corner_name",
        "g_min_mean",
        "g_max_mean",
        "g_hole_depth",
        "g_util_mean",
        "g_hole_phase",
        "severity",
        "interpretation",
    ]
    display_df = g_hole_summary[display_cols].rename(
        columns={
            "corner_name": "Corner",
            "g_min_mean": "Min G (avg)",
            "g_max_mean": "Max G (avg)",
            "g_hole_depth": "G Hole Depth",
            "g_util_mean": "G Util %",
            "g_hole_phase": "Phase",
            "severity": "Severity",
            "interpretation": "Interpretation",
        }
    )

    print("G Hole Detection Summary (sorted by depth, worst first):")
    display(
        display_df.style.format(
            {
                "Min G (avg)": "{:.2f}",
                "Max G (avg)": "{:.2f}",
                "G Hole Depth": "{:.2f}",
                "G Util %": "{:.1f}",
            }
        )
    )

    # Call out high-severity corners
    high = g_hole_summary[g_hole_summary["severity"] == "HIGH"]
    if len(high) > 0:
        print(f"\n⚠ {len(high)} corner(s) with HIGH severity G holes:")
        for _, row in high.iterrows():
            print(
                f"  {row['corner_name']}: G drops to {row['g_min_mean']:.2f}G "
                f"(from {row['g_max_mean']:.2f}G peak) during {row['g_hole_phase']} phase"
            )
else:
    print("No G utilization data available for G hole detection")

In [ ]:
# Summary statistics table for all braking metrics
def compute_braking_summary(stats_df):
    """Compute summary statistics for braking metrics across all laps."""
    summary = []
    metrics = [
        ("peak_brake", "Peak Brake Pressure"),
        ("entry_speed", "Entry Speed (km/h)"),
        ("braking_distance", "Braking Distance (m)"),
        ("brake_release_point", "Brake Release Point (m)"),
        ("braking_point", "Braking Point (m)"),
    ]

    for seg_name in stats_df[stats_df["segment_type"] == "braking"]["segment_name"].unique():
        for col, label in metrics:
            seg_data = stats_df[(stats_df["segment_name"] == seg_name) & stats_df[col].notna()]
            if len(seg_data) > 0:
                summary.append(
                    {
                        "Segment": seg_name,
                        "Metric": label,
                        "Mean": seg_data[col].mean(),
                        "Std": seg_data[col].std(),
                        "Min": seg_data[col].min(),
                        "Max": seg_data[col].max(),
                        "Range": seg_data[col].max() - seg_data[col].min(),
                        "N": len(seg_data),
                    }
                )

    return pd.DataFrame(summary)


braking_summary_df = compute_braking_summary(stats_df)
if len(braking_summary_df) > 0:
    braking_summary_df.style.format(
        {
            "Mean": "{:.1f}",
            "Std": "{:.1f}",
            "Min": "{:.1f}",
            "Max": "{:.1f}",
            "Range": "{:.1f}",
        }
    )
else:
    print("No braking data available for summary")